# 1. Remove rows belonging to certain industries (make this clean)

In [ ]:
run_overwrite = True

import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

error_industries = ['43', '47', '74']

dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(dirs.db_path))

# TODO: search for industries without fame_derived
table_fame_derived = con.table("fame_derived")
table_fame_fixed = con.table("fame_fixed")
table_fame_yearly = con.table("fame_yearly")

# Helper function to prevent repeating the same counting block 3 times
def get_table_counts(derived, fixed, yearly, count_col_name="count"):
    return pd.DataFrame({
        "table": ["fame_derived", "fame_fixed", "fame_yearly"],
        count_col_name: [derived.count().execute(), fixed.count().execute(), yearly.count().execute()]
    })

count_before_df = get_table_counts(table_fame_derived, table_fame_fixed, table_fame_yearly, "count_before")
print(f"✅ Counts before filtering:\n{count_before_df.to_markdown(index=False)}\n")

# ==========================================
# 1. FILTERED (Firms IN error_industries)
# ==========================================
table_fame_derived_filtered = table_fame_derived.filter(table_fame_derived.industry_codes.isin(error_industries))

# SEMI-JOIN: Keeps rows in the left table that have a match in the right table.
# This eliminates the need for an inner_join + manual .select()
table_fame_fixed_filtered = table_fame_fixed.semi_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_filtered = table_fame_yearly.semi_join(table_fame_derived_filtered, "registered_number")

count_errors_df = get_table_counts(table_fame_derived_filtered, table_fame_fixed_filtered, table_fame_yearly_filtered, "count_after")
print(f"✅ Counts after filtering:\n{count_errors_df.to_markdown(index=False)}\n")

# ==========================================
# 2. EXCLUDED (Firms NOT IN error_industries)
# ==========================================
table_fame_derived_excluded = table_fame_derived.filter(~table_fame_derived.industry_codes.isin(error_industries))

# ANTI-JOIN: Keeps rows in the left table that DO NOT have a match in the right table.
# This replaces the cumbersome left_join + isnull() filter + select()
table_fame_fixed_excluded = table_fame_fixed.anti_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_excluded = table_fame_yearly.anti_join(table_fame_derived_filtered, "registered_number")

count_excluded_df = get_table_counts(table_fame_derived_excluded, table_fame_fixed_excluded, table_fame_yearly_excluded, "count_excluded")
print(f"✅ Counts after exclusion:\n{count_excluded_df.to_markdown(index=False)}\n")

# ==========================================
# 3. VERIFICATION
# ==========================================
# Use Pandas vectorized math to check all three tables simultaneously
expected_counts = count_before_df["count_before"] - count_errors_df["count_after"]
matches = count_excluded_df["count_excluded"] == expected_counts

if not matches.all():
    failed_tables = count_excluded_df.loc[~matches, "table"].tolist()
    raise ValueError(f"❌ Excluded counts do not match expected counts for: {', '.join(failed_tables)}")

for _, row in count_excluded_df.iterrows():
    t_name = row["table"]
    cb_val = int(count_before_df.loc[count_before_df['table'] == t_name, 'count_before'].sum()) # type: ignore
    ca_val = int(count_errors_df.loc[count_errors_df['table'] == t_name, 'count_after'].sum()) # type: ignore
    print(f"✅ Excluded counts match expected counts for table {t_name}: {row['count_excluded']:,} = "
          f"{cb_val:,} - {ca_val:,}")

# Output a sample of 10 rows from each of the filtered tables for verification
# Refactor the below: if any table has a zero error count then don't display
count_derived = count_errors_df.loc[count_errors_df['table'] == 'fame_derived', 'count_after'].sum()
if count_derived > 0:
    sample_frac_derived = 10 / count_derived
    display(table_fame_derived_filtered.sample(sample_frac_derived).execute())
count_fixed = count_errors_df.loc[count_errors_df['table'] == 'fame_fixed', 'count_after'].sum()
if count_fixed > 0:
    sample_frac_fixed = 10 / count_fixed
    display(table_fame_fixed_filtered.sample(sample_frac_fixed).execute())
count_yearly = count_errors_df.loc[count_errors_df['table'] == 'fame_yearly', 'count_after'].sum()
if count_yearly > 0:
    sample_frac_yearly = 10 / count_yearly
    display(table_fame_yearly_filtered.sample(sample_frac_yearly).execute())

# ==========================================
# 4. SAFE DATABASE OVERWRITE (ATOMIC SWAP)
# ==========================================
if run_overwrite:
    print("💾 Writing clean data to temporary tables and performing atomic swaps...")

    # 1. Update fame_derived
    # Check if the excluded count is zero
    if count_derived > 0:
        con.create_table("fame_derived_clean", table_fame_derived_excluded, overwrite=True)
        con.drop_table("fame_derived")
        con.create_table("fame_derived", con.table("fame_derived_clean"), overwrite=True)
        print("✅ Successfully updated fame_derived table.")
    else:
        print("⚠️ Excluded count for fame_derived is zero. Skipping update to fame_derived table.")

    # 2. Update fame_fixed
    if count_fixed > 0:
        con.create_table("fame_fixed_clean", table_fame_fixed_excluded, overwrite=True)
        con.drop_table("fame_fixed")
        con.create_table("fame_fixed", con.table("fame_fixed_clean"), overwrite=True)
        print("✅ Successfully updated fame_fixed table.")
    else:
        print("⚠️ Excluded count for fame_fixed is zero. Skipping update to fame_fixed table.")

    # 3. Update fame_yearly
    if count_yearly > 0:
        con.create_table("fame_yearly_clean", table_fame_yearly_excluded, overwrite=True)
        con.drop_table("fame_yearly")
        con.create_table("fame_yearly", con.table("fame_yearly_clean"), overwrite=True)
        print("✅ Successfully updated fame_yearly table.")
    else:
        print("⚠️ Excluded count for fame_yearly is zero. Skipping update to fame_yearly table.")

✅ Counts before filtering:
| table        |   count_before |
|:-------------|---------------:|
| fame_derived |        7948736 |
| fame_fixed   |        7948736 |
| fame_yearly  |       40846539 |

✅ Counts after filtering:
| table        |   count_after |
|:-------------|--------------:|
| fame_derived |        235756 |
| fame_fixed   |        314169 |
| fame_yearly  |        298244 |

✅ Counts after exclusion:
| table        |   count_excluded |
|:-------------|-----------------:|
| fame_derived |          7712980 |
| fame_fixed   |          7634567 |
| fame_yearly  |         40548295 |

✅ Excluded counts match expected counts for table fame_derived: 7,712,980 = 7,948,736 - 235,756
✅ Excluded counts match expected counts for table fame_fixed: 7,634,567 = 7,948,736 - 314,169
✅ Excluded counts match expected counts for table fame_yearly: 40,548,295 = 40,846,539 - 298,244


,registered_number,has_ptaddress,has_ptaddress_latlong,is_public,has_company_branch_mismatch,industry_codes,file_codes
0,10155897,True,False,False,False,74,18_59 2
1,10466444,True,False,False,False,74,18_59 2
2,11429679,True,False,False,False,74,19_00 1
3,11489516,True,True,False,False,74,19_00 1
4,13687587,False,False,False,None,74,19_01
5,13909382,False,False,False,False,74,19_01
6,03487894,False,False,False,None,74,18_57
7,10766460,True,False,False,False,74,19_00
8,SC392626,False,False,False,None,74,19_02 1


,company_name,registered_number,ticker_symbol,ro_address,ro_address_line_1,ro_address_line_2,ro_address_line_3,ro_address_line_4,ro_address_line_5,ro_city,...,primary_trading_address_latitude,primary_trading_address_longitude,branch_name,primary_uk_sic_2007_code,primary_uk_sic_2007_description,latest_accounts_date,no_of_available_years,guo,guo_nb,entity_type
0,HUNTERS RECRUITMENT CONSULTANCY LIMITED,12927121,NaN,"10 Salisbury Court, Newmarket Avenue, Northolt...",10 Salisbury Court,Newmarket Avenue,NaN,NaN,NaN,Northolt,...,NaN,NaN,HUNTERS RECRUITMENT CONSULTANCY LIMITED,74909,"Other professional, scientific and technical a...",2021-10-31,1,NaN,0,Single location
1,ANNA MCPHERSON DESIGN LIMITED,10180834,NaN,"Raithby House, Raithby, Spilsby, Lincolnshire,...",Raithby House,Raithby,NaN,NaN,NaN,Spilsby,...,"53° 10' 41.7"" N","0° 3' 19.9"" E",ANNA MCPHERSON DESIGN LIMITED,74100,Specialised design activities,2020-05-31,4,NaN,0,Single location
2,BLUEFEATHER STUDIOS LIMITED,10415705,NaN,"52 Grenada Crescent, Newton Leys, Bletchley, M...",52 Grenada Crescent,Newton Leys,Bletchley,NaN,NaN,Milton Keynes,...,"51° 55' 23.0"" N","0° 10' 29.7"" W",BLUEFEATHER STUDIOS LIMITED,59112,Video production activities,2023-10-31,7,MR SURESH SEETHARAMAN,2,Controlled subs.
3,NEW GENERATION FUELS LIMITED,11438695,NaN,"1 Royal Terrace, Southend-On-Sea, Essex, SS1 1EA",1 Royal Terrace,NaN,NaN,NaN,NaN,Southend-On-Sea,...,NaN,NaN,NaN,74909,"Other professional, scientific and technical a...",2022-06-30,4,NaN,0,Single location
4,NANERGY (UK) LIMITED,14346054,NaN,"Flat 26, Ashley Court, Chapelfields, Frodsham,...",Flat 26,Ashley Court,Chapelfields,NaN,NaN,Frodsham,...,NaN,NaN,NANERGY (UK) LIMITED,74901,Environmental consulting activities,2024-09-30,2,CONNELL ALEXANDER,2,Controlled subs.
5,PLANETBODS LIMITED,14385509,NaN,"9b Broadfield, High Roding, Dunmow, Essex, CM6...",9b Broadfield,High Roding,NaN,NaN,NaN,Dunmow,...,NaN,NaN,PLANETBODS LIMITED,74100,Specialised design activities,2023-09-30,1,NaN,0,Single location
6,VISIONARY HUB SPACE LIMITED,10687360,NaN,"Dbh 9 Diss Business Park, Hopper Way, Diss, No...",Dbh 9 Diss Business Park,Hopper Way,NaN,NaN,NaN,Diss,...,NaN,NaN,NaN,73110,Advertising agencies,2019-03-31,2,NaN,0,Single location
7,EFALIUS LTD,08399763,NaN,"85 Water Lane, Middlestown, Wakefield, West Yo...",85 Water Lane,Middlestown,NaN,NaN,NaN,Wakefield,...,"53° 39' 14.9"" N","1° 34' 54.4"" W",EFALIUS LTD,74909,"Other professional, scientific and technical a...",2014-02-28,1,NaN,0,Single location
8,KHROMA INTERIOR DESIGN LIMITED,08574063,NaN,"95 Plumstead Road, Norwich, Norfolk, NR1 4JS",95 Plumstead Road,NaN,NaN,NaN,NaN,Norwich,...,"52° 38' 5.7"" N","1° 19' 15.6"" E",KHROMA INTERIOR DESIGN LIMITED,47599,"Retail sale of furniture, lighting equipment a...",2014-06-30,1,NaN,0,Single location


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,08441883,2014,False,NaN,0.001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12999495,2021,False,NaN,0.100,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,08164721,2019,False,NaN,0.001,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07777521,2014,False,NaN,36.856,NaN,NaN,0.410,NaN,NaN,...,NaN,0.554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12577904,2021,False,NaN,0.001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,09360134,2023,False,NaN,1181.583,NaN,11.0,533.451,348.159,348.159,...,NaN,39.630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,12389083,2021,False,NaN,0.002,NaN,2.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


💾 Writing clean data to temporary tables and performing atomic swaps...
✅ Successfully updated fame_derived table.
✅ Successfully updated fame_fixed table.
✅ Successfully updated fame_yearly table.


# 2. Migrate: drop table fame_derived and merge into fame_fixed

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

data_dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(data_dirs.db_path))

table_fame_fixed = con.table("fame_fixed")
table_fame_derived = con.table("fame_derived")

# Safely cast to a list to avoid Tuple + List TypeErrors
# 0. Rename the columns to plural before processing to avoid confusion
# 1. Isolate the columns we need and drop exact duplicates
cols_fame_fixed = list(table_fame_fixed.columns)
table_fame_derived = table_fame_derived.rename(
    industry_code = "industry_codes",
    file_code = "file_codes"
)
table_fame_derived_skinny = table_fame_derived.select(
    "registered_number", "industry_code", "file_code"
).distinct()

# 2. Collapse any remaining multiple rows for the same registered_number 
# into comma-separated strings to guarantee a 1-to-1 join
count_skinny = table_fame_derived_skinny.count().execute()
print(f"🔍 Collapsing {count_skinny:,} rows for the same registered_number into comma-separated strings...")
table_fame_derived_collapsed = table_fame_derived_skinny.group_by("registered_number").aggregate(
    industry_codes=table_fame_derived_skinny.industry_code.group_concat(","),
    file_codes=table_fame_derived_skinny.file_code.group_concat(",")
)
count_collapsed = table_fame_derived_collapsed.count().execute()
print(f"✅ Collapsed to {count_collapsed:,} unique registered_number rows from fame_derived.")

# 3. Perform the left join
# Because the right table now has strictly unique keys, this will NEVER fan out.
table_fame_merged = table_fame_fixed.left_join(table_fame_derived_collapsed, "registered_number")

# 4. Select the exact final schema to clean up any join artifacts
table_fame_merged_skinny = table_fame_merged.select(cols_fame_fixed + ["industry_codes", "file_codes"])

# --- VERIFICATION ---
num_rows = table_fame_merged_skinny.count().execute()
fixed_rows = table_fame_fixed.count().execute()

print(f"✅ Number of rows in fame_merged_skinny: {num_rows:,}")
print(f"✅ Number of rows in fame_fixed: {fixed_rows:,}")

if num_rows != fixed_rows:
    raise ValueError("❌ Row count mismatch! The number of rows in fame_merged_skinny does not match fame_fixed.")

print("✅ Schema of fame_merged_skinny:")
print(table_fame_merged_skinny.schema())

🔍 Collapsing 9,047,723 rows for the same registered_number into comma-separated strings...
✅ Collapsed to 8,114,908 unique registered_number rows from fame_derived.
✅ Number of rows in fame_merged_skinny: 9,283,479
✅ Number of rows in fame_fixed: 9,283,479
✅ Schema of fame_merged_skinny:
ibis.Schema {
  company_name                       string
  registered_number                  string
  ticker_symbol                      string
  ro_address                         string
  ro_address_line_1                  string
  ro_address_line_2                  string
  ro_address_line_3                  string
  ro_address_line_4                  string
  ro_address_line_5                  string
  ro_city                            string
  ro_county                          string
  ro_postcode                        string
  ro_full_postcode                   string
  ro_country                         string
  ro_latitude                        string
  ro_longitude                       

In [2]:
# Safe overwrite
print("💾 Writing fame_merged_skinny to temporary table fame_fixed_clean...")
con.create_table("fame_fixed_clean", table_fame_merged_skinny, overwrite=True)
print("💾 Executing 4-step safe table overwrite procedure...")
con.drop_table("fame_fixed")
con.create_table("fame_fixed", con.table("fame_fixed_clean"), overwrite=True)
con.drop_table("fame_fixed_clean")
print("✅ Successfully updated fame_fixed table with merged and collapsed data.")

con.drop_table("fame_derived")
print("✅ Successfully dropped fame_derived table after merging into fame_fixed.")

💾 Writing fame_merged_skinny to temporary table fame_fixed_clean...
💾 Executing 4-step safe table overwrite procedure...
✅ Successfully updated fame_fixed table with merged and collapsed data.
✅ Successfully dropped fame_derived table after merging into fame_fixed.


# 3. Migrate to new db file to save space

### Copy a table over from a backup

In [ ]:
# Copy every table in interesting_tables to a new duckdb file
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

con_old = ibis.duckdb.connect(str("E:\\Database backups\\fame_data_v5_filter_8m_47m.duckdb"))
con_new = ibis.duckdb.connect(str(dirs.db_path))
table_name = "fame_fixed"
rename_to = "fame_fixed_filtered"

old_tables = con_old.list_tables()
new_tables = con_new.list_tables()
print(old_tables)
print(new_tables)

# Safely overwrite in con_new fame_fixed -> con_new fame_fixed_filtered (just renaming a table)
if "fame_fixed" in new_tables:
    try:
        assert con_new.table(table_name).count().execute() == 152379
        con_new.create_table(rename_to, con_new.table(table_name), overwrite=True)
        assert con_new.table(rename_to).count().execute() == 152379
        con_new.drop_table(table_name)
    except Exception as e:
        print(f"❌ Error during {table_name} -> {rename_to} rename: {e}")

# Copy over the fame_fixed table from con_old to con_new as fame_fixed.
new_tables_after = con_new.list_tables()
try:
    old_fixed = con_old.table(table_name)
    old_fixed_count = old_fixed.count().execute()
    print(f"Count: {old_fixed_count:,}")
    assert old_fixed.count().execute() == 9283479
    con_new.create_table(table_name, old_fixed.execute(), overwrite=table_name in new_tables_after)
    assert con_new.table(table_name).count().execute() == 9283479
except Exception as e:
    print(f"❌ Error during copying {table_name} table: {e}")

['fame_derived_clean', 'fame_fixed', 'fame_yearly_clean', 'fame_yearly_consolidated', 'ibis_duckdb_table_wb3dz4ibdzdnlal634nm3kgh3i', 'lars_fixed', 'lars_yearly', 'working_yearly_kp']
['fame_fixed', 'fame_fixed_filtered', 'fame_yearly', 'fame_yearly_filtered', 'ibis_duckdb_table_r5bvmyv3qvbdnjqhb3iyxswu2u', 'ibis_duckdb_table_sbogedwhnzainohiqf6v6ixmwe', 'lars_fixed', 'lars_yearly']
❌ Error during fame_fixed -> fame_fixed_filtered rename: 
Count: 9,283,479


### Migrate table using parquet export/import
- Takes about 5 minutes per database, for a big database.

In [3]:
import os
import shutil
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")

db_dir = dirs.output_dir
dbs_to_backup = ["fame_data.duckdb"]
interesting_tables = [
    "fame_fixed_filtered", "fame_yearly_kp",
    "working_fixed", "working_yearly", "working_yearly_with_tfp_wave",
    "working_distance_pc4", "working_distance_ttwa5km",
    "ref_ons_postcode", "ref_ons_ttwa_name"
]
# 0. Set parameters
# If dbs_to_backup is a dict, assume it is a key-value pair of the [old_db_name, new_db_name]
# If it is a list, create a dict with the with the new names being old name with "_backup" appended
dbs_to_backup_dict = {}
if isinstance(dbs_to_backup, dict):
    dbs_to_backup_dict = dbs_to_backup
elif isinstance(dbs_to_backup, list):
    dbs_to_backup_dict = {db_name: f"{os.path.splitext(db_name)[0]}_backup.duckdb" for db_name in dbs_to_backup}

for old_db_name, new_db_name in dbs_to_backup_dict.items():
    print(old_db_name, "->", new_db_name)
    export_dir = str(db_dir / f"{os.path.splitext(old_db_name)[0]}_export_temp")
    old_db_path = db_dir / old_db_name
    new_db_path = db_dir / new_db_name

    # 0. OS get the file size of the old database for logging
    old_db_size = os.path.getsize(old_db_path)

    # # 1. Export the old database
    print(f"📤 Exporting old database to \"{export_dir}\"...")
    con_old = ibis.duckdb.connect(str(old_db_path), read_only=True)
    con_old.raw_sql("PRAGMA memory_limit='12GB'")
    con_old.raw_sql(f"EXPORT DATABASE '{export_dir}' (FORMAT PARQUET);")

    # 2. Connect to the pristine new database
    con_new = ibis.duckdb.connect(str(new_db_path))
    con_new_tables = con_new.list_tables()      # Should be empty if the new database is pristine
    print(f"Tables in new database: {con_new_tables}")

    # 3. Scan the exported directory and load ONLY the matching Parquet files
    for table_name in interesting_tables:

        # Parquet has a quirk where any numbers in the table_name will be replaced by underscore (_). Check for that
        parquet_path = os.path.join(export_dir, f"{table_name}.parquet")
        u_table_name = ''.join(['_' if c.isdigit() else c for c in table_name])
        u_parquet_path = os.path.join(export_dir, f"{u_table_name}.parquet")

        if os.path.exists(parquet_path):
            print(f"--- 📥 Importing {table_name} into new database...")
            # read_parquet allows Ibis to stream the Parquet file directly into the new table
            con_new.create_table(table_name, con_new.read_parquet(parquet_path), overwrite=table_name in con_new_tables)

        elif os.path.exists(u_parquet_path):
            print(f"--- 📥 Importing {u_table_name} into new database...")
            con_new.create_table(u_table_name, con_new.read_parquet(u_parquet_path), overwrite=u_table_name in con_new_tables)
            interesting_tables[interesting_tables.index(table_name)] = u_table_name

        else:
            print(f"--- ⚠️ Skipping {table_name}: Not found in export directory.")

    # 4. Drop non-relevant tables
    for table_name in con_new.list_tables():
        if table_name in interesting_tables:
            continue
        try:
            print(f"--- Dropping table {table_name} from new database...")
            con_new.drop_table(table_name)
        except Exception as e:
            print(f"⚠️ Couldn't drop table {table_name}: {e}")
            try:
                print(f"🗑️ Attempting to drop view: {table_name}")
                con_new.drop_view(table_name)  # Attempt to drop view if it exists
            except Exception as e:
                print(f"❌ Failed to handle {table_name} as both table and view: {e}")

    # 5. Print list of tables and rows in the new database for verification
    df_new_tables = con_new.list_tables()
    df_new_rows = {t: con_new.table(t).count().execute() for t in df_new_tables}
    df_new_tr = pd.DataFrame(list(df_new_rows.items()), columns=["table_name", "row_count"]).sort_values("table_name")
    print(f"Tables in db2 ({new_db_name}):")
    display(df_new_tr)

    new_db_size = os.path.getsize(new_db_path)
    # print the size reduction in GB
    print(f"--- Reduced size from {old_db_size / (1024**3):.2f} GB to {new_db_size / (1024**3):.2f} GB ({(1 - new_db_size / old_db_size) * 100:.2f}%).")
    # 6. Clean up the export payload
    shutil.rmtree(export_dir, ignore_errors=True)
    con_old.disconnect()
    con_new.raw_sql("CHECKPOINT;")
    con_new.disconnect()

fame_data.duckdb -> fame_data_backup.duckdb
📤 Exporting old database to "C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data_export_temp"...
Tables in new database: []
--- 📥 Importing fame_fixed_filtered into new database...
--- 📥 Importing fame_yearly_kp into new database...
--- 📥 Importing working_fixed into new database...
--- 📥 Importing working_yearly into new database...
--- 📥 Importing working_yearly_with_tfp_wave into new database...
--- 📥 Importing working_distance_pc_ into new database...
--- 📥 Importing working_distance_ttwa_km into new database...
--- 📥 Importing ref_ons_postcode into new database...
--- 📥 Importing ref_ons_ttwa_name into new database...
--- Dropping table ibis_read_parquet_4avbqfuftnb5nkdhtofrr7ygkm from new database...
⚠️ Couldn't drop table ibis_read_parquet_4avbqfuftnb5nkdhtofrr7ygkm: Catalog Error: Existing object ibis_read_parquet_4avbqfuftnb5nkdhtofrr7ygkm is of type View, trying to drop type Table
🗑️ Attempting to drop view: ibis_read_parq

,table_name,row_count
0,fame_fixed_filtered,152379
1,fame_yearly_kp,1128490
2,ref_ons_postcode,2726477
3,ref_ons_ttwa_name,230
4,working_distance_pc_,43241404
5,working_distance_ttwa_km,734346954
6,working_fixed,152379
7,working_yearly,1128490
8,working_yearly_with_tfp_wave,1081520


--- Reduced size from 16.15 GB to 8.06 GB (50.06%).


In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")

db_dir = dirs.output_dir
new_db_path = db_dir / "fame_data.duckdb"
con_new = ibis.duckdb.connect(str(new_db_path))
con_new_tables = con_new.list_tables()
print(f"Tables in new database: {con_new_tables}")

Tables in new database: ['fame_fixed_filtered', 'fame_yearly_filtered', 'fame_yearly_kp', 'lars_fixed', 'lars_yearly', 'ref_ons_postcode', 'ref_ons_ttwa_name', 'working_distance', 'working_distance_filtered', 'working_fixed', 'working_yearly', 'working_yearly_with_peers']


In [4]:
drop_tables = ["lars_fixed", "lars_yearly", "fame_yearly_filtered"]
rename_tables = {
    "working_distance": "working_distance_ttwa5km",
    "working_distance_filtered": "working_distance_pc4"
}

for table_name in drop_tables:
    if table_name in con_new_tables:
        con_new.drop_table(table_name)
        print(f"✅ Dropped table {table_name}.")
    else:
        print(f"⚠️ Table {table_name} not found in new database. Skipping drop.")
for old_name, new_name in rename_tables.items():
    if old_name in con_new_tables:
        con_new.create_table(new_name, con_new.table(old_name), overwrite=True)
        con_new.drop_table(old_name)
        print(f"✅ Renamed table {old_name} to {new_name}.")
    else:
        print(f"⚠️ Table {old_name} not found in new database. Skipping rename.")

✅ Dropped table lars_fixed.
✅ Dropped table lars_yearly.
✅ Dropped table fame_yearly_filtered.
✅ Renamed table working_distance to working_distance_ttwa5km.
✅ Renamed table working_distance_filtered to working_distance_pc4.
